# EDA - NYC Taxi

In [ ]:
import os
import sys

if "__file__" in globals():
    HERE = os.path.dirname(os.path.abspath(__file__))
else:
    HERE = os.getcwd()

SRC_ROOT = os.path.abspath(os.path.join(HERE, "..", "src"))
if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

from common.config import (
    MONTHS,
    SCHEMA_BRONZE,
    SCHEMA_GOLD,
    SCHEMA_SILVER,
    STUDY_END_EXCLUSIVE,
    STUDY_START_DATE,
    get_landing_month_path,
)

dbutils.widgets.text("catalog", "ifood")
catalog = dbutils.widgets.get("catalog")

yellow_bronze = f"{catalog}.{SCHEMA_BRONZE}.yellow_trips"
green_bronze = f"{catalog}.{SCHEMA_BRONZE}.green_trips"
yellow_quarantine = f"{catalog}.{SCHEMA_SILVER}.yellow_trips_quarantine"
green_quarantine = f"{catalog}.{SCHEMA_SILVER}.green_trips_quarantine"
fact_trip = f"{catalog}.{SCHEMA_GOLD}.fact_trip"

spark.sql(
    f"""
    CREATE OR REPLACE TEMP VIEW bronze_union AS
    SELECT
        'yellow' AS taxi_type,
        VendorID,
        passenger_count,
        trip_distance,
        RatecodeID,
        PULocationID,
        DOLocationID,
        payment_type,
        fare_amount,
        total_amount,
        airport_fee,
        tpep_pickup_datetime AS pickup_datetime,
        tpep_dropoff_datetime AS dropoff_datetime,
        _ingestion_ts,
        _source_file,
        _batch_id,
        _rescued_data
    FROM {yellow_bronze}

    UNION ALL

    SELECT
        'green' AS taxi_type,
        VendorID,
        passenger_count,
        trip_distance,
        RatecodeID,
        PULocationID,
        DOLocationID,
        payment_type,
        fare_amount,
        total_amount,
        airport_fee,
        lpep_pickup_datetime AS pickup_datetime,
        lpep_dropoff_datetime AS dropoff_datetime,
        _ingestion_ts,
        _source_file,
        _batch_id,
        _rescued_data
    FROM {green_bronze}
    """
)

spark.sql(
    f"""
    CREATE OR REPLACE TEMP VIEW silver_quarantine_union AS
    SELECT
        'yellow' AS taxi_type,
        VendorID,
        passenger_count,
        total_amount,
        pickup_datetime,
        dropoff_datetime,
        _ingestion_ts,
        _quarantined_at,
        _rejection_reasons
    FROM {yellow_quarantine}

    UNION ALL

    SELECT
        'green' AS taxi_type,
        VendorID,
        passenger_count,
        total_amount,
        pickup_datetime,
        dropoff_datetime,
        _ingestion_ts,
        _quarantined_at,
        _rejection_reasons
    FROM {green_quarantine}
    """
)

landing_schema_rows = []
for year, month in MONTHS:
    year_month = f"{year}-{month:02d}"
    for taxi_type in ("yellow", "green"):
        month_path = get_landing_month_path(catalog, taxi_type, year, month)
        schema = spark.read.parquet(month_path).schema
        for field in schema.fields:
            landing_schema_rows.append(
                (
                    taxi_type,
                    year_month,
                    field.name,
                    field.dataType.simpleString(),
                    field.nullable,
                )
            )

landing_schema_df = spark.createDataFrame(
    landing_schema_rows,
    [
        "taxi_type",
        "year_month",
        "column_name",
        "data_type",
        "nullable",
    ],
)
landing_schema_df.createOrReplaceTempView("landing_schema_snapshot")
spark.table(fact_trip).createOrReplaceTempView("fact_trip_view")

print(f"catalog={catalog}")
print(f"study_range=[{STUDY_START_DATE}, {STUDY_END_EXCLUSIVE})")


## 1. Os arquivos de origem possuem schema consistente entre os meses?

In [ ]:
%sql
SELECT
  taxi_type,
  column_name,
  COUNT(DISTINCT data_type) AS distinct_types,
  SORT_ARRAY(COLLECT_SET(CONCAT(year_month, ' -> ', data_type))) AS observed_types
FROM landing_schema_snapshot
GROUP BY taxi_type, column_name
HAVING COUNT(DISTINCT data_type) > 1
ORDER BY taxi_type, column_name


Schema por mês

In [ ]:
%sql
SELECT
  taxi_type,
  year_month,
  column_name,
  data_type,
  nullable
FROM landing_schema_snapshot
ORDER BY taxi_type, year_month, column_name


## 2. Quais campos críticos apresentam nulos na Bronze?

Foco nos campos `VendorID`, `passenger_count`, `RatecodeID`, `PULocationID`, `DOLocationID` e `payment_type`.


In [ ]:
%sql
SELECT
  taxi_type,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_rows,
  SUM(CASE WHEN VendorID IS NULL THEN 1 ELSE 0 END) AS vendorid_nulls,
  SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS passenger_count_nulls,
  SUM(CASE WHEN RatecodeID IS NULL THEN 1 ELSE 0 END) AS ratecodeid_nulls,
  SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS pulocationid_nulls,
  SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS dolocationid_nulls,
  SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type_nulls
FROM bronze_union
GROUP BY taxi_type
ORDER BY taxi_type


## 3. Os nulos em campos críticos estão concentrados em linhas com `_rescued_data`?

In [ ]:
%sql
SELECT
  taxi_type,
  COUNT(*) AS rescued_rows,
  SUM(CASE WHEN VendorID IS NULL THEN 1 ELSE 0 END) AS vendorid_nulls,
  SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS passenger_count_nulls,
  SUM(CASE WHEN RatecodeID IS NULL THEN 1 ELSE 0 END) AS ratecodeid_nulls,
  SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS pulocationid_nulls,
  SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS dolocationid_nulls,
  SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type_nulls
FROM bronze_union
WHERE _rescued_data IS NOT NULL
GROUP BY taxi_type
ORDER BY taxi_type


Query para verificar se o campo vem nulo da origem

In [ ]:
%sql
SELECT
  taxi_type,
  VendorID,
  passenger_count,
  RatecodeID,
  PULocationID,
  DOLocationID,
  payment_type,
  _source_file,
  _rescued_data
FROM bronze_union
WHERE _rescued_data IS NOT NULL
  AND (
    VendorID IS NULL
    OR passenger_count IS NULL
    OR RatecodeID IS NULL
    OR PULocationID IS NULL
    OR DOLocationID IS NULL
    OR payment_type IS NULL
  )
ORDER BY taxi_type, _source_file
LIMIT 50


## 4. `passenger_count` nulo ? problema da origem ou problema no processamento?

In [ ]:
%sql
SELECT
  taxi_type,
  COUNT(*) AS rows_with_null_passenger_count,
  SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_rows_with_null_passenger_count
FROM bronze_union
WHERE passenger_count IS NULL
GROUP BY taxi_type
ORDER BY taxi_type


Query para detalhar a análise

In [ ]:
%sql
SELECT
  taxi_type,
  pickup_datetime,
  dropoff_datetime,
  VendorID,
  passenger_count,
  total_amount,
  _source_file,
  _rescued_data
FROM bronze_union
WHERE passenger_count IS NULL
ORDER BY taxi_type, pickup_datetime
LIMIT 50


## 5. Há registros fora do intervalo do case?

In [ ]:
%sql
SELECT
  taxi_type,
  COUNT(*) AS rows_outside_study_range,
  MIN(pickup_datetime) AS min_pickup_datetime,
  MAX(pickup_datetime) AS max_pickup_datetime
FROM bronze_union
WHERE pickup_datetime < TIMESTAMP '2023-01-01'
   OR pickup_datetime >= TIMESTAMP '2023-06-01'
GROUP BY taxi_type
ORDER BY taxi_type


Temos datas anomalas?

In [ ]:
%sql
SELECT
  taxi_type,
  MIN(pickup_datetime) AS min_pickup_datetime,
  MAX(pickup_datetime) AS max_pickup_datetime
FROM bronze_union
GROUP BY taxi_type
ORDER BY taxi_type


## 6. Verificar outliers ou valores suspeitos

In [ ]:
%sql
SELECT
  taxi_type,
  MIN(passenger_count) AS min_passenger_count,
  PERCENTILE_APPROX(passenger_count, 0.5) AS p50_passenger_count,
  PERCENTILE_APPROX(passenger_count, 0.95) AS p95_passenger_count,
  PERCENTILE_APPROX(passenger_count, 0.99) AS p99_passenger_count,
  MAX(passenger_count) AS max_passenger_count
FROM bronze_union
WHERE passenger_count IS NOT NULL
GROUP BY taxi_type
ORDER BY taxi_type


In [ ]:
%sql
SELECT
  taxi_type,
  MIN(total_amount) AS min_total_amount,
  ROUND(PERCENTILE_APPROX(total_amount, 0.5), 2) AS p50_total_amount,
  ROUND(PERCENTILE_APPROX(total_amount, 0.95), 2) AS p95_total_amount,
  ROUND(PERCENTILE_APPROX(total_amount, 0.99), 2) AS p99_total_amount,
  MAX(total_amount) AS max_total_amount
FROM bronze_union
WHERE total_amount IS NOT NULL
GROUP BY taxi_type
ORDER BY taxi_type


In [ ]:
%sql
SELECT
  taxi_type,
  MIN(trip_distance) AS min_trip_distance,
  ROUND(PERCENTILE_APPROX(trip_distance, 0.5), 2) AS p50_trip_distance,
  ROUND(PERCENTILE_APPROX(trip_distance, 0.95), 2) AS p95_trip_distance,
  ROUND(PERCENTILE_APPROX(trip_distance, 0.99), 2) AS p99_trip_distance,
  MAX(trip_distance) AS max_trip_distance
FROM bronze_union
WHERE trip_distance IS NOT NULL
GROUP BY taxi_type
ORDER BY taxi_type


In [ ]:
%sql
SELECT
  taxi_type,
  pickup_datetime,
  dropoff_datetime,
  passenger_count,
  trip_distance,
  total_amount,
  _source_file
FROM bronze_union
WHERE passenger_count IS NOT NULL
ORDER BY passenger_count DESC, total_amount DESC
LIMIT 25


In [ ]:
%sql
SELECT
  taxi_type,
  pickup_datetime,
  dropoff_datetime,
  passenger_count,
  trip_distance,
  total_amount,
  _source_file
FROM bronze_union
WHERE total_amount IS NOT NULL
ORDER BY total_amount DESC, trip_distance DESC
LIMIT 25


## 7. O que foi para quarentena na Silver e por qual motivo?

In [ ]:
%sql
SELECT
  taxi_type,
  _rejection_reasons,
  COUNT(*) AS rows_in_quarantine
FROM silver_quarantine_union
GROUP BY taxi_type, _rejection_reasons
ORDER BY taxi_type, rows_in_quarantine DESC


In [ ]:
%sql
SELECT
  taxi_type,
  COUNT(*) AS quarantine_rows,
  MIN(pickup_datetime) AS min_pickup_datetime,
  MAX(pickup_datetime) AS max_pickup_datetime
FROM silver_quarantine_union
GROUP BY taxi_type
ORDER BY taxi_type


Exemplos

In [ ]:
%sql
SELECT
  taxi_type,
  VendorID,
  passenger_count,
  total_amount,
  pickup_datetime,
  dropoff_datetime,
  _rejection_reasons
FROM silver_quarantine_union
ORDER BY taxi_type, pickup_datetime
LIMIT 50


## 8. A Gold preserva o recorte analático esperado?

In [ ]:
%sql
SELECT
  taxi_type,
  date_format(pickup_datetime, 'yyyy-MM') AS year_month,
  COUNT(*) AS trips
FROM fact_trip_view
GROUP BY taxi_type, date_format(pickup_datetime, 'yyyy-MM')
ORDER BY taxi_type, year_month


In [ ]:
%sql
SELECT
  taxi_type,
  MIN(pickup_datetime) AS min_pickup_datetime,
  MAX(pickup_datetime) AS max_pickup_datetime,
  MIN(total_amount) AS min_total_amount,
  MAX(total_amount) AS max_total_amount
FROM fact_trip_view
GROUP BY taxi_type
ORDER BY taxi_type


In [ ]:
%sql
SELECT
  HOUR(pickup_datetime) AS pickup_hour,
  COUNT(*) AS trips,
  ROUND(AVG(passenger_count), 4) AS avg_passenger_count
FROM fact_trip_view
WHERE MONTH(pickup_datetime) = 5
GROUP BY HOUR(pickup_datetime)
ORDER BY pickup_hour
